# ClimateScope:Visualizing Global Weather Trends and Extreme Events 
## Project objective
The objective of ClimateScope is to analyze and visually represent global weather patterns 
using the Global Weather Repository dataset. This project aims to uncover seasonal trends, 
regional variations, and extreme weather events through interactive and insightful visualizations. 
By leveraging daily-updated, worldwide weather data, the project will enable users to explore 
climate behavior over time, compare conditions across regions, and identify anomalies. The 
ultimate goal is to provide an accessible, data-driven platform that supports climate awareness, 
decision-making, and further research into global weather dynamics.

# Milestone 2: Core Analysis & Visualization Design

## Objective
In this milestone, we will perform statistical analysis on the cleaned weather dataset
to understand:
- Distributions (how values like temperature and humidity are spread)
- Correlations (how variables move together)
- Seasonal patterns (repeating cycles by month)
- Trends (long-term changes over time)


In [ ]:
# Importing required libraries

import pandas as pd             # For handling datasets (tables)
import numpy as np              # For mathematical operations
import matplotlib.pyplot as plt # For plotting graphs
import seaborn as sns           # For making beautiful statistical plots
from statsmodels.tsa.seasonal import seasonal_decompose #helps to find seasonal patterns and trends mathematically.


In [ ]:
#loading dataset
data=pd.read_csv("CleanedWeatherRepository.csv")
dataMonth=pd.read_csv("CleanedWeatherRepositoryMonthly.csv")

In [ ]:
data.head()

In [ ]:
dataMonth.head()

### Why we loaded two datasets
- `weather_clean.csv` → contains detailed daily/hourly weather observations.
  We’ll use it for **distributions, correlations, and extreme events.**
- `weather_monthly_avg.csv` → contains pre-aggregated monthly averages.
  We’ll use it for **seasonal and trend analysis** (faster and cleaner).

Having both datasets helps us analyze both **short-term patterns** (daily)
and **long-term changes** (monthly/yearly).


In [ ]:
data.info()

In [ ]:
data.describe().round(2)

### Understanding the Dataset
- `info()` shows how many rows are non-null (to check if there are missing values).
- `describe()` gives statistical summary:
  - **mean** → average value
  - **std** → how spread out data is
  - **min / max** → lowest & highest readings
  - **25%, 50%, 75%** → quartiles (help to see distribution spread)


#### Statistical Analysis

In [ ]:
# -----------------------------------------------
#  Select relevant columns for analysis
# -----------------------------------------------

numeric_cols = [
    'temperature_celsius',
    'feels_like_celsius',
    'humidity',
    'wind_kph',
    'gust_kph',
    'pressure_mb',
    'precip_mm',
    'air_quality_PM2.5',
    'air_quality_PM10'
]

selected_cols = ['country', 'location_name', 'date', 'month'] + numeric_cols
dataSelected = data[selected_cols]

print("Selected columns for Milestone 2 analysis:")
print(dataSelected.columns.tolist())


In [ ]:

dataSelected.info()

### What Each Metric Means
| Metric | Description | Why it Matters |
|---------|--------------|----------------|
| **Mean** | The average of all values | General average temperature or humidity |
| **Median** | Middle value (when sorted) | More stable when there are outliers |
| **Mode** | Most frequent value | Shows common recurring reading |
| **Min / Max** | Lowest & highest | Shows extremes |
| **Std Dev** | Standard deviation — how much variation exists | High std dev → unstable weather |
| **Variance** | Square of std dev | Statistical measure of variability |
| **Range** | Max − Min | Overall spread of data |


In [ ]:

# Creating a statistical summary table
stats_data = pd.DataFrame({
    'Mean': data[numeric_cols].mean().round(2),
    'Median': data[numeric_cols].median().round(2),
    'Min': data[numeric_cols].min().round(2),
    'Max': data[numeric_cols].max().round(2),
    'Std Dev': data[numeric_cols].std().round(2),
    'Variance': data[numeric_cols].var().round(2),
    'Range': (data[numeric_cols].max() - data[numeric_cols].min()).round(2)
})

stats_data


**comparing Mean vs Median**

In [ ]:
import plotly.graph_objects as go

#  Bright color palette — energetic yet elegant
bright_mean = '#00B4D8'   # vivid aqua blue
bright_median = '#FF6F61' # coral orange-red

fig = go.Figure()

# Mean bars
fig.add_trace(go.Bar(
    x=stats_data.index,
    y=stats_data['Mean'],
    name='Mean',
    marker_color=bright_mean
))

# Median bars
fig.add_trace(go.Bar(
    x=stats_data.index,
    y=stats_data['Median'],
    name='Median',
    marker_color=bright_median
))

# Layout customization for vibrant style
fig.update_layout(
    title=dict(
        text=' Mean vs Median Comparison for Weather Variables',
        x=0.5,
        font=dict(size=24, color='#222831', family='Verdana')
    ),
    xaxis_title='Variables',
    yaxis_title='Value',
    barmode='group',
    template='plotly_white',
    plot_bgcolor='rgba(250,250,250,1)',
    paper_bgcolor='rgba(255,255,255,1)',
    legend=dict(
        title='Statistic',
        orientation='h',
        y=-0.25,
        x=0.3,
        bgcolor='rgba(0,0,0,0)',
        font=dict(size=13)
    ),
    xaxis=dict(showgrid=False, tickfont=dict(size=12, color='#333')),
    yaxis=dict(showgrid=True, gridcolor='rgba(200,200,200,0.3)'),
)

# Add soft shadow for 3D-like pop
fig.update_traces(marker_line_width=1.5, marker_line_color='rgba(0,0,0,0.1)')

fig.show()


If mean ≈ median, the data is normal (no extreme spikes).

If mean >> median, it’s right-skewed (few very high values, e.g., rainfall).

If mean << median, it’s left-skewed (few very low values).
### Insights
- **Temperature & Pressure** → Mean ≈ Median → balanced, stable readings.
- **Wind & Precipitation** → Mean > Median → right-skewed → occasional strong storms or heavy rains.
- **Humidity** → close match → moderate variability.

**Standard Deviation Bar Chart — Show Variable Stability**

This helps visually compare which variables fluctuate the most.

In [ ]:
import plotly.express as px
import pandas as pd
# Compute standard deviations for all numeric columns
std_values = data[numeric_cols].std().sort_values(ascending=False)

# Convert to DataFrame for Plotly
std_data = pd.DataFrame({
    'Variable': std_values.index,
    'Standard Deviation': std_values.values
})

# Use a bright modern color palette (not same color for all bars)
colors = px.colors.qualitative.Vivid  # includes bright varied shades

# Create Plotly bar chart
fig = px.bar(
    std_data,
    x='Variable',
    y='Standard Deviation',
    text='Standard Deviation',
    color='Variable',
    color_discrete_sequence=colors,
    title=' Standard Deviation of All Numeric Weather Variables'
)

# Styling
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(
    template='plotly_white',
    title_font=dict(size=24, color='#222831', family='Verdana'),
    xaxis_title='Variables',
    yaxis_title='Standard Deviation',
    plot_bgcolor='rgba(250,250,250,1)',
    paper_bgcolor='rgba(255,255,255,1)',
    xaxis=dict(showgrid=False, tickangle=45, tickfont=dict(size=10)),
    yaxis=dict(showgrid=True, gridcolor='rgba(200,200,200,0.3)'),
    showlegend=False
)

fig.show()

###  Standard Deviation of All Numeric Weather Variables

**Purpose:**  
To measure how much each numeric weather variable fluctuates from its mean.  
Higher standard deviation = higher variability (unstable factor).  
Lower standard deviation = more consistent (stable factor).


**Simple Summary Visualization**

This helps visualize which variables have large spreads around their mean (unstable) and which are tight (stable).

In [ ]:

# Calculate Mean and Standard Deviation
summary_data = pd.DataFrame({
    'Mean': data[numeric_cols].mean().round(2),
    'Std Dev': data[numeric_cols].std().round(2)
}).reset_index().rename(columns={'index': 'Variable'})

# ✨ Classy color combination
color_mean = '#FFD700'   # golden yellow
color_std  = '#6A0DAD'   # royal purple

fig = go.Figure()

# Mean bars
fig.add_trace(go.Bar(
    name='Mean',
    x=summary_data['Variable'],
    y=summary_data['Mean'],
    marker_color=color_mean,
    text=summary_data['Mean'],
    textposition='outside'
))

# Standard deviation bars
fig.add_trace(go.Bar(
    name='Standard Deviation',
    x=summary_data['Variable'],
    y=summary_data['Std Dev'],
    marker_color=color_std,
    text=summary_data['Std Dev'],
    textposition='outside'
))

# 💎 Layout Styling
fig.update_layout(
    title=dict(
        text=' Mean and Standard Deviation of Weather Variables',
        x=0.5,
        font=dict(size=24, color='#0A043C', family='Verdana')
    ),
    xaxis_title='Variables',
    yaxis_title='Value',
    barmode='group',
    template='plotly_white',
    plot_bgcolor='rgba(250,250,250,1)',
    paper_bgcolor='rgba(255,255,255,1)',
    xaxis=dict(showgrid=False, tickangle=45, tickfont=dict(size=10, color='#1B263B')),
    yaxis=dict(showgrid=True, gridcolor='rgba(200,200,200,0.3)'),
    legend=dict(
        title='Statistic',
        orientation='h',
        y=-0.25,
        x=0.3,
        bgcolor='rgba(0,0,0,0)',
        font=dict(size=13, color='#0A043C')
    )
)

# Add subtle border for definition
fig.update_traces(marker_line_width=1.3, marker_line_color='rgba(0,0,0,0.15)')

fig.show()

##  Mean and Standard Deviation of Weather Variables

###  Purpose
To compare the **average (Mean)** and **variability (Standard Deviation)** of every numeric weather variable in the dataset.  
This gives a balanced overview of **stability vs fluctuation** across temperature, wind, pressure, humidity, and other indicators.

This bar chart illustrates the **mean** and **standard deviation** for key weather-related variables such as temperature, humidity, wind speed, and air quality.

- **Temperature and Feels Like (°C):**  
  Both variables have similar mean values (~22–24°C), indicating that the actual temperature and the perceived temperature are closely aligned. The relatively low standard deviation shows that temperature readings remain fairly consistent over time.

- **Humidity (%):**  
  The mean humidity is around 64%, with moderate variability (standard deviation ≈ 24%). This suggests that humidity levels fluctuate but generally stay within a comfortable range.

- **Wind and Gust Speed (kph):**  
  The average wind speed is 13.27 kph, while gusts average around 18.56 kph. The higher standard deviation in gust speed (≈ 14.68) reflects more irregular wind bursts compared to the overall wind speed.

- **Pressure (mb):**  
  The pressure variable shows a high mean value (≈ 1016 mb) with a small standard deviation, indicating **stable atmospheric pressure** conditions with minimal variation.

- **Precipitation (mm):**  
  The mean precipitation is very low (≈ 0.14 mm), while the standard deviation (≈ 0.6 mm) suggests occasional rainfall spikes amidst generally dry conditions.

- **Air Quality (PM2.5 and PM10):**  
  Air quality metrics show higher standard deviations, particularly PM10 (mean ≈ 53.77, SD ≈ 164.17). This indicates **significant fluctuations in air pollution levels**, possibly due to seasonal or environmental factors.

**Insight:**  
Overall, the weather data reveals stable temperature and pressure patterns but high variability in air quality and wind gusts. This can be important when analyzing how atmospheric or pollution-related factors influence other variables or events (e.g., traffic or health conditions).


In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Compute statistics (Min and Max only)
summary_data = pd.DataFrame({
    'Min': data[numeric_cols].min().round(2),
    'Max': data[numeric_cols].max().round(2)
}).reset_index().rename(columns={'index': 'Variable'})

#  New color palette
color_min = '#FFD700'   # bright yellow
color_max = '#6A0DAD'   # royal purple

# Create grouped bar chart
fig = go.Figure(data=[
    go.Bar(name='Min', x=summary_data['Variable'], y=summary_data['Min'],
           marker_color=color_min, text=summary_data['Min'], textposition='outside'),
    go.Bar(name='Max', x=summary_data['Variable'], y=summary_data['Max'],
           marker_color=color_max, text=summary_data['Max'], textposition='outside')
])

# 💎 Layout customization
fig.update_layout(
    title=dict(
        text=' Min and Max of Weather Variables',
        x=0.5,
        font=dict(size=24, color='#0A043C', family='Verdana')
    ),
    xaxis_title='Variables',
    yaxis_title='Value',
    barmode='group',
    template='plotly_white',
    plot_bgcolor='rgba(250,250,250,1)',
    paper_bgcolor='rgba(255,255,255,1)',
    xaxis=dict(showgrid=False, tickangle=45, tickfont=dict(size=10, color='#1B263B')),
    yaxis=dict(showgrid=True, gridcolor='rgba(200,200,200,0.3)'),
    legend=dict(
        title='Statistic',
        orientation='h',
        y=-0.25,
        x=0.3,
        bgcolor='rgba(0,0,0,0)',
        font=dict(size=13, color='#0A043C')
    )
)

# Subtle borders for clarity
fig.update_traces(marker_line_width=1.3, marker_line_color='rgba(0,0,0,0.15)')

fig.show()


## Distribution Analysis
A distribution shows how often each value appears in dataset.
It answers questions like:

Are most temperatures around 25 °C, or spread across many values?

Do we have a few extremely high rainfall days?

Is humidity evenly distributed or mostly high?

Think of it as the “shape” of our data — whether it’s:

Normal (bell-shaped)

Right-skewed (many small values, few large ones)

Left-skewed (many large values, few small ones)

**Goal of Distribution Analysis**

Understand the “shape” of each variable	->Is data balanced or extreme?
Detect outliers ->	Unusual values stand out in plots
Identify skewness ->	Helps decide transformations for modeling later
Support insights visually ->	Add charts that make our report more intuitive

In [ ]:
import plotly.express as px
import plotly.colors as colors

# Define unique beautiful gradients for each variable
gradient_sets = {
    'temperature_celsius': ['#FF6B6B', '#FFA07A', '#FFD93D'],  # warm sunset (red → coral → gold)
    'humidity': ['#4ECDC4', '#44A8A0', '#95E1D3'],  # tropical ocean (teal → turquoise → mint)
    'wind_kph': ['#A8E6CF', '#7FCDCD', '#6C9FE8'],  # breezy sky (mint → cyan → sky blue)
    'pressure_mb': ['#9B59B6', '#8E44AD', '#BE90D4'],  # royal amethyst (purple variations)
    'precip_mm': ['#3498DB', '#5DADE2', '#85C1E9']  # rainfall blue (deep blue → light blue)
}

# Columns to visualize
key_cols = ['temperature_celsius', 'humidity', 'wind_kph', 'pressure_mb', 'precip_mm']

for col in key_cols:
    fig = px.histogram(
        data, 
        x=col,
        nbins=40,
        marginal="box",   # adds boxplot on top
        title=f" Distribution of {col}",
        color_discrete_sequence=gradient_sets[col]
    )
    
    fig.update_traces(marker=dict(
        line=dict(width=0.5, color='rgba(0,0,0,0.2)'),
        opacity=0.85
    ))
    
    fig.update_layout(
        template='plotly_white',
        title_font=dict(size=22, color='#0A043C', family='Verdana'),
        xaxis_title=col,
        yaxis_title="Frequency",
        plot_bgcolor='rgba(250,250,250,1)',
        paper_bgcolor='rgba(255,255,255,1)',
        font=dict(color='#1B263B'),
        bargap=0.05,
    )
    
    fig.show()

### Distribution of Temperature (°C)
Most temperature values fall between **20°C and 30°C**, showing generally warm conditions.  
A few outliers on both ends indicate rare extreme cold or hot readings.

### Distribution of Humidity (%)
Humidity values are mostly between **40% and 80%**, showing moderate to high moisture levels.  
Some low and high outliers reflect dry or highly humid weather events.

### Distribution of Wind Speed (kph)
Most wind speeds are below **20 kph**, suggesting calm to mild wind conditions.  
A few higher values indicate occasional strong wind occurrences.

### Distribution of Pressure (mb)
Pressure readings are concentrated around **1000–1020 mb**, typical for stable atmospheric conditions.  
Slight deviations suggest minor weather variations.

### Distribution of Precipitation (mm)
Most data points show **low or zero rainfall**, meaning many dry periods.  
A few high values indicate occasional heavy rainfall events.


**Histogram + KDE Plot**

A histogram shows frequency; the KDE curve (blue line) shows smooth density.

In [ ]:
# -----------------------------------------------
# Fixed: Distribution plots (no empty subplot)
# -----------------------------------------------
cols = [
    'temperature_celsius', 'humidity',
    'wind_kph', 'pressure_mb', 'precip_mm'
]

fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(12, 10))
axes = axes.flatten()

for i, col in enumerate(cols):
    sns.histplot(data[col], bins=30, kde=True, color='skyblue', ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

# 🔹 Hide any extra unused subplot
for j in range(len(cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


### Histogram + KDE Interpretation
- The **bars** show how frequently each range of values occurs.
- The **blue curve (KDE)** shows the smooth shape of the distribution.
- A **tall, narrow peak** → values are consistent (low variation).
- A **wide, flat shape** → large variation.
- A **right-skewed** curve (tail on the right) → few high values, many small (e.g., rainfall).
### Distribution Plots of Weather Attributes

The histograms below show how key weather parameters are distributed in the dataset:

- **Temperature (°C):** Most values lie between **20°C and 30°C**, indicating warm conditions with few extreme readings.  
- **Humidity (%):** Concentrated between **40% and 80%**, showing generally moderate to high humidity.  
- **Wind Speed (kph):** Mostly below **20 kph**, suggesting calm weather with occasional strong winds.  
- **Pressure (mb):** Clustered around **1000–1020 mb**, typical for stable atmospheric pressure.  
- **Precipitation (mm):** Mostly near **0**, meaning many dry days with a few heavy rainfall events.

Overall, the plots highlight that the dataset captures typical weather variations with occasional extremes across different parameters.


In [ ]:
import plotly.express as px

# Key numeric columns to visualize
key_cols = ['temperature_celsius', 'humidity', 'wind_kph', 'pressure_mb', 'precip_mm']

fig = px.violin(
    data,
    y=key_cols,
    box=True,                 # show the boxplot inside
    points="outliers",        # show individual outlier points
    color_discrete_sequence=['#6A0DAD', '#C71585', '#FFD700', '#00B4D8', '#1B263B'],  # royal purple to navy gradient
    title="🎻 Distribution & Spread of Weather Variables (Violin Plot)"
)

fig.update_layout(
    template='plotly_white',
    title_font=dict(size=24, color='#0A043C', family='Verdana'),
    yaxis_title="Value",
    xaxis_title="Variables",
    plot_bgcolor='rgba(250,250,250,1)',
    paper_bgcolor='rgba(255,255,255,1)'
)
fig.show()


## 🎻 Violin Plot — Combining Shape & Spread

### 📘 Why We Use It
A **violin plot** merges two key ideas:
- It shows the **distribution shape** (density).
- It shows the **data spread** and **outliers** (like a boxplot inside).

The *width* of each violin tells us how common certain values are:
- **Wide parts** → many observations.
- **Narrow parts** → fewer observations.
- **Dots outside** → outliers.
### Violin Plot — Distribution & Spread of Weather Variables

This violin plot compares the **distribution and spread** of all key weather attributes.  
- **Temperature** and **humidity** show moderate spread with few extreme outliers.  
- **Wind speed** values are mostly low, with a few higher readings.  
- **Pressure** has the smallest variation, staying around a narrow range.  
- **Precipitation** is heavily skewed toward zero, showing many dry observations.

Overall, the plot highlights both the **central tendency and variability** of each weather parameter in a single view.


In [ ]:
import plotly.express as px

# Compute standard deviation for all numeric columns
std_values = data[key_cols].std().sort_values(ascending=False)
std_data = pd.DataFrame({'Variable': std_values.index, 'Std Dev': std_values.values})

fig = px.bar(
    std_data,
    x='Variable',
    y='Std Dev',
    color='Variable',
    color_discrete_sequence=['#FFD700', '#6A0DAD', '#00B4D8', '#FF6F61', '#1B263B'],
    title=" Variable Stability vs Volatility (Based on Standard Deviation)"
)

fig.update_traces(texttemplate='%{y:.2f}', textposition='outside')
fig.update_layout(
    template='plotly_white',
    title_font=dict(size=22, color='#0A043C', family='Verdana'),
    xaxis_title="Weather Variables",
    yaxis_title="Standard Deviation",
    plot_bgcolor='rgba(250,250,250,1)',
    paper_bgcolor='rgba(255,255,255,1)',
    showlegend=False
)
fig.show()


##  Identifying Stable vs Volatile Variables

###  Why We Use This
The **standard deviation** measures how much a variable fluctuates:
- **Low Std Dev → Stable Variable** (values are close to the mean)
- **High Std Dev → Volatile Variable** (values vary widely)
### Variable Stability vs Volatility (Standard Deviation)

This bar chart shows how much each weather variable fluctuates in the dataset.  
- Higher bars indicate **greater variability** (less stable), while lower bars show **more consistent values**.  
- **Temperature** and **wind speed** usually have higher standard deviations, meaning they change more often.  
- **Pressure** and **humidity** are relatively stable, showing smaller variations.  
- **Precipitation** varies the least, with most days having little or no rainfall.

Overall, this visualization helps identify which weather factors are **more volatile** and which remain **consistent** across observations.


#### What is Correlation?

Correlation tells us how strongly two variables are related.

Correlation Value

+1	Perfect positive relationship	

0	No relationship	

–1	Perfect negative relationship

In [ ]:

# Compute correlation matrix
corr_matrix = data[numeric_cols].corr().round(2)

# Display first few correlations
corr_matrix.head()


In [ ]:
import plotly.figure_factory as ff

# Create annotated heatmap
fig = ff.create_annotated_heatmap(
    z=corr_matrix.values,
    x=list(corr_matrix.columns),
    y=list(corr_matrix.index),
    colorscale='Cividis',  # you can change to 'Plasma', 'Cividis',Viridis etc.
    showscale=True,
    annotation_text=corr_matrix.values.round(2)
)

fig.update_layout(
    title=dict(
        text='🌍 Correlation Heatmap of Weather Variables',
        x=0.5,
        font=dict(size=24, color='#0A043C', family='Verdana')
    ),
    template='plotly_white',
    width=950,
    height=950,
)
fig.show()


##  Correlation Heatmap — Understanding Relationships Between Variables

###  Why We Use It
A **correlation heatmap** visually shows how pairs of variables move together:
- Bright colors → strong correlation (positive or negative)
- Pale colors → weak or no correlation

This helps identify patterns and dependencies between weather factors.
### Correlation Heatmap of Weather Variables

This heatmap visualizes the **correlation** between all key weather parameters.  
### Correlation Interpretation
The heatmap shows how variables move together.

- **Temperature vs Pressure** → slight negative correlation (warm air = lower pressure).
- **Humidity vs Precipitation** → positive correlation (humid → rainy).
- **Wind vs Precipitation** → moderate correlation (storms).

Correlation analysis helps identify cause-effect patterns in climate variables.

Overall, the heatmap helps identify how weather features are **interconnected** and which ones influence each other most.


In [ ]:
key_features = ['temperature_celsius', 'humidity', 'wind_kph', 'pressure_mb', 'precip_mm']

fig = ff.create_annotated_heatmap(
    z=data[key_features].corr().values,
    x=key_features,
    y=key_features,
    colorscale='Plasma',
    showscale=True,
    annotation_text=data[key_features].corr().round(2).values
)

fig.update_layout(
    title=dict(text='🔍 Focused Correlation Heatmap (Key Weather Variables)',
               x=0.5, font=dict(size=22, color='#0A043C')),
    template='plotly_white',
    width=800,
    height=800,
)
fig.show()


#### Seasonal Trends & Comparative Analysis 🎯

This step transforms our cleaned dataset into insights about time and geography —
we’ll see how temperature, humidity, and rainfall change across months and countries, using Plotly visualizations that are both interactive and beautiful.

##### What Does This Mean?

Let’s first understand what we’re trying to find here:

Concept -->	Meaning	-->Example

**Seasonal Patterns**
How weather variables change through months/seasons	

Temperature rises in summer, drops in winter

**Trends**

Overall direction of change (increasing/decreasing)	

Gradual warming trend over time

**Comparative Analysis**

Compare regions, countries, or climates

Which country has higher rainfall?

We’ll use your monthly cleaned dataset (CleanedWeatherRepositoryMonthly.csv)
since it already contains year, month, and average weather values — perfect for seasonal trends.

In [ ]:
import plotly.express as px

# Average temperature per month (global)
monthly_temp = dataMonth.groupby('month')['temperature_celsius'].mean().reset_index()

fig = px.line(
    monthly_temp,
    x='month',
    y='temperature_celsius',
    markers=True,
    color_discrete_sequence=['#FF6F61'],
    title=" Average Global Temperature by Month"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Avg Temperature (°C)",
    template='plotly_white',
    title_font=dict(size=22, color='#0A043C'),
    plot_bgcolor='rgba(250,250,250,1)'
)
fig.show()


####  Monthly Temperature Trends

#### Why We Use It
Line charts reveal how **temperature fluctuates month by month**, showing **seasonal variation**.


In [ ]:
top_countries = dataMonth['country'].value_counts().nlargest(5).index  # pick top 5 countries
subset = dataMonth[dataMonth['country'].isin(top_countries)]

fig = px.line(
    subset,
    x='month',
    y='temperature_celsius',
    color='country',
    markers=True,
    line_shape='spline',
    title="🌍 Monthly Temperature Comparison Across Top 5 Countries",
    color_discrete_sequence=['#6A0DAD', '#00B4D8', '#FF6F61', '#FFD700', '#1B263B']
)

fig.update_layout(
    template='plotly_white',
    xaxis_title="Month",
    yaxis_title="Temperature (°C)",
    title_font=dict(size=22, color='#0A043C')
)
fig.show()


####  Comparing Temperature Trends Across Countries

#####  Why We Use It
Multi-line plots help us see **how weather differs across countries** through the year.
### Monthly Temperature Comparison Across Top 5 Countries

This line chart compares the **average monthly temperature trends** for the top five countries.  
- Each line represents one country’s temperature changes over time.  
- The **smooth curves** show seasonal patterns — some countries experience clear rises and drops, while others stay relatively steady.  
- **Temperature peaks** and **dips** highlight warm and cold months across regions.  

Overall, this visualization helps compare **seasonal variations** and understand how climate patterns differ between countries.


#### Humidity Trends
 ##### Global Monthly Average

In [ ]:
import plotly.express as px
import pandas as pd

# Average humidity per month
monthly_humidity = dataMonth.groupby('month')['humidity'].mean().reset_index()

fig = px.line(
    monthly_humidity,
    x='month',
    y='humidity',
    markers=True,
    color_discrete_sequence=['#00B4D8'],
    title=" Average Global Humidity by Month"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Average Humidity (%)",
    template='plotly_white',
    title_font=dict(size=22, color='#0A043C')
)
fig.show()


#### Top 5 Country Comparison

In [ ]:
top_countries = dataMonth['country'].value_counts().nlargest(5).index
subset = dataMonth[dataMonth['country'].isin(top_countries)]

fig = px.line(
    subset,
    x='month',
    y='humidity',
    color='country',
    markers=True,
    line_shape='spline',
    color_discrete_sequence=['#00B4D8', '#6A0DAD', '#FFD700', '#FF6F61', '#1B263B'],
    title="💧 Monthly Humidity Comparison Across Top 5 Countries"
)
fig.show()


### Monthly Humidity Comparison Across Top 5 Countries

This line chart shows the **monthly humidity trends** for the top five countries.  
- Each line represents one country’s variation in humidity throughout the year.  
- Some countries show **consistent humidity**, while others experience **strong seasonal fluctuations**.  
- Peaks indicate **more humid months**, while dips show **drier periods**.  

Overall, this visualization highlights how **moisture levels vary seasonally** across different regions.


#### Precipitation Trends
 ##### Global Monthly Average

In [ ]:
monthly_precip = dataMonth.groupby('month')['precip_mm'].mean().reset_index()

fig = px.line(
    monthly_precip,
    x='month',
    y='precip_mm',
    markers=True,
    color_discrete_sequence=['#6A0DAD'],
    title=" Average Global Precipitation by Month"
)
fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Average Precipitation (mm)",
    template='plotly_white',
)
fig.show()


#### Top 5 Country Comparison

In [ ]:
fig = px.line(
    subset,
    x='month',
    y='precip_mm',
    color='country',
    markers=True,
    line_shape='spline',
    color_discrete_sequence=['#6A0DAD', '#FFB703', '#219EBC', '#FB8500', '#8ECAE6'],
    title=" Monthly Precipitation Comparison Across Top 5 Countries"
)
fig.show()


### Monthly Precipitation Comparison Across Top 5 Countries

This line chart shows the **monthly rainfall patterns** for the top five countries.  
- Each line represents one country’s **average precipitation** over the months.  
- Noticeable **peaks** indicate months with heavy rainfall, while **flat or low segments** show drier periods.  
- The variation across countries reflects different **climatic and seasonal rainfall patterns**.  

Overall, this plot helps compare how **rainfall intensity and timing** differ across regions.


#### Wind Speed Trends
##### Global Monthly Average

In [ ]:
monthly_wind = dataMonth.groupby('month')['wind_kph'].mean().reset_index()

fig = px.line(
    monthly_wind,
    x='month',
    y='wind_kph',
    markers=True,
    color_discrete_sequence=['#FF6F61'],
    title="💨 Average Global Wind Speed by Month"
)
fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Average Wind Speed (kph)",
    template='plotly_white',
)
fig.show()


#### Top 5 Country Comparison

In [ ]:
fig = px.line(
    subset,
    x='month',
    y='wind_kph',
    color='country',
    markers=True,
    line_shape='spline',
    color_discrete_sequence=['#FF6F61', '#00B4D8', '#6A0DAD', '#FFD700', '#1B263B'],
    title="💨 Monthly Wind Speed Comparison Across Top 5 Countries"
)
fig.show()


### Monthly Wind Speed Comparison Across Top 5 Countries

This line chart illustrates the **monthly wind speed trends** for the top five countries.  
- Each line represents the variation in **average wind speed** throughout the year.  
- Some countries show **steady wind patterns**, while others have noticeable **spikes** during certain months.  
- Peaks may indicate **stormy or windy seasons**, whereas dips suggest **calmer weather**.  

Overall, this chart helps compare **seasonal wind behavior** across different countries.


#### Pressure Trends
#### Global Monthly Average

In [ ]:
monthly_pressure = dataMonth.groupby('month')['pressure_mb'].mean().reset_index()

fig = px.line(
    monthly_pressure,
    x='month',
    y='pressure_mb',
    markers=True,
    color_discrete_sequence=['#FFD700'],
    title="⚖️ Average Global Pressure by Month"
)
fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Average Pressure (mb)",
    template='plotly_white',
)
fig.show()


##### Top 5 Country Comparison

In [ ]:
fig = px.line(
    subset,
    x='month',
    y='pressure_mb',
    color='country',
    markers=True,
    line_shape='spline',
    color_discrete_sequence=['#FFD700', '#6A0DAD', '#00B4D8', '#FF6F61', '#1B263B'],
    title=" Monthly Pressure Comparison Across Top 5 Countries"
)
fig.show()


### Monthly Pressure Comparison Across Top 5 Countries

This line chart shows the **monthly atmospheric pressure trends** for the top five countries.  
- Each line represents one country’s **average pressure** variation throughout the year.  
- Most countries show **minor fluctuations**, indicating relatively stable pressure levels.  
- Slight peaks or dips may reflect **seasonal weather shifts** or **regional climate differences**.  

Overall, this plot highlights how **air pressure remains mostly consistent** with small variations across months and countries.


In [ ]:
import plotly.express as px
import pandas as pd

# Select pollutant columns
pollutants = [
    'air_quality_Carbon_Monoxide', 'air_quality_Nitrogen_dioxide',
    'air_quality_Sulphur_dioxide', 'air_quality_Ozone',
    'air_quality_PM2.5', 'air_quality_PM10'
]

# Compute monthly averages for each pollutant
monthly_air = dataMonth.groupby('month')[pollutants].mean().reset_index()

# Melt the data to long format for Plotly
air_melted = monthly_air.melt(id_vars='month', var_name='Pollutant', value_name='Concentration')

# Multi-line plot
fig = px.line(
    air_melted,
    x='month',
    y='Concentration',
    color='Pollutant',
    markers=True,
    line_shape='spline',
    title=" Monthly Trends of All Air Pollutants",
    color_discrete_sequence=['#FF6F61', '#FFD700', '#6A0DAD', '#00B4D8', '#1B263B', '#FFB703']
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Concentration (units vary by pollutant)",
    template='plotly_white',
    title_font=dict(size=22, color='#0A043C'),
    plot_bgcolor='rgba(250,250,250,1)',
)
fig.show()


Compare **average pollutant levels by country** side-by-side.

### Monthly Trends of All Air Pollutants

This line chart displays the **average monthly concentration** of key air pollutants.  
- Each line represents one pollutant’s trend across months.  
- Some pollutants, like **Ozone** and **Nitrogen Dioxide**, show clear **seasonal variations**.  
- Others, such as **PM2.5** and **PM10**, maintain relatively consistent levels with occasional peaks.  
- The chart highlights how **air quality changes throughout the year** depending on pollutant type and environmental conditions.  

Overall, it provides an overview of **pollution dynamics** and helps identify **months with higher pollutant levels**.


In [ ]:
# Average per country
country_air = dataMonth.groupby('country')[pollutants].mean().reset_index()
top_countries = dataMonth['country'].value_counts().nlargest(5).index
country_air = country_air[country_air['country'].isin(top_countries)]

# Melt for Plotly
country_air_melted = country_air.melt(id_vars='country', var_name='Pollutant', value_name='Concentration')

fig = px.bar(
    country_air_melted,
    x='country',
    y='Concentration',
    color='Pollutant',
    barmode='group',
    title=" Comparison of Air Pollutants Across Top 5 Countries",
    color_discrete_sequence=['#FF6F61', '#FFD700', '#6A0DAD', '#00B4D8', '#1B263B', '#FFB703']
)

fig.update_layout(
    xaxis_title="Country",
    yaxis_title="Average Concentration",
    template='plotly_white',
    title_font=dict(size=22, color='#0A043C')
)
fig.show()


In [ ]:
import plotly.figure_factory as ff

# Compute monthly averages
monthly_air_matrix = dataMonth.groupby('month')[pollutants].mean().T

fig = ff.create_annotated_heatmap(
    z=monthly_air_matrix.values,
    x=list(range(1, 13)),
    y=list(monthly_air_matrix.index),
    colorscale='Plasma',
    annotation_text=monthly_air_matrix.round(1).values,
    showscale=True
)

fig.update_layout(
    title=" Monthly Air Pollutant Concentrations (Heatmap)",
    template='plotly_white',
    title_font=dict(size=22, color='#0A043C'),
    width=900,
    height=600
)
fig.show()


####  Air Quality Heatmap (All Pollutants)

#####  Why We Use This
A heatmap is perfect for spotting **which months have higher pollution** for each pollutant.


#### Air Quality Choropleth Map (All Air Parameters)

We’ll visualize multiple pollutants —
like CO, NO₂, SO₂, O₃, PM2.5, and PM10 — one at a time using dropdown filters.

### Monthly Air Pollutant Concentrations (Heatmap)

This heatmap visualizes the **average concentration of each air pollutant** across the months.  
- **Darker shades** represent **higher pollutant levels**, while lighter shades indicate cleaner air.  
- Each row corresponds to a specific pollutant, showing how its concentration changes monthly.  
- Noticeable seasonal peaks highlight months with **poorer air quality** for certain pollutants.  

Overall, this visualization makes it easy to compare **pollution intensity and seasonal trends** across all air quality indicators.


In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Compute mean pollutant levels per country
pollutants = [
    'air_quality_Carbon_Monoxide', 'air_quality_Nitrogen_dioxide',
    'air_quality_Sulphur_dioxide', 'air_quality_Ozone',
    'air_quality_PM2.5', 'air_quality_PM10'
]

air_map_data = dataMonth.groupby('country')[pollutants].mean().reset_index()

# Create initial choropleth (default pollutant)
fig = go.Figure()

fig.add_trace(go.Choropleth(
    locations=air_map_data['country'],
    locationmode='country names',
    z=air_map_data['air_quality_PM2.5'],
    colorscale='Turbo',
    colorbar_title="PM2.5 (µg/m³)",
))

# Add dropdown menu for pollutant selection
fig.update_layout(
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label=pollutant.replace("air_quality_", "").replace("_", " "),
                    method='update',
                    args=[
                        {'z': [air_map_data[pollutant]]},
                        {'colorbar.title.text': pollutant.replace("air_quality_", "").replace("_", " ")}
                    ]
                )
                for pollutant in pollutants
            ],
            direction='down',
            showactive=True,
            x=0.05,
            y=1.15,
            xanchor='left',
            yanchor='top'
        )
    ],
    title=dict(
        text=" Global Air Quality Levels (Select Pollutant)",
        x=0.5,
        font=dict(size=22, color='#0A043C')
    ),
    geo=dict(showframe=False, showcoastlines=True, projection_type='natural earth'),
    template='plotly_white',
)
fig.show()


### Global Air Quality Levels (Interactive Map)

This interactive choropleth map displays the **average air pollutant concentrations** for each country.  
- Use the **dropdown menu** to switch between pollutants like **PM2.5**, **Ozone**, and **Nitrogen Dioxide**.  
- **Darker regions** indicate **higher pollution levels**, while lighter shades show cleaner air.  
- The map helps visualize **geographical differences** in air quality and identify countries with higher pollutant concentrations.  

Overall, it provides an intuitive global view of **air quality distribution** across multiple pollutants.


#### Weather Parameter Choropleth Map

Now, let’s make one for Temperature, Humidity, Pressure, and Precipitation —
these together define climate conditions.

In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Compute average weather parameters per country
weather_params = ['temperature_celsius', 'humidity', 'pressure_mb', 'precip_mm']
weather_data = dataMonth.groupby('country')[weather_params].mean().reset_index()

# Create base figure (default: temperature)
fig = go.Figure()

fig.add_trace(go.Choropleth(
    locations=weather_data['country'],
    locationmode='country names',
    z=weather_data['temperature_celsius'],
    colorscale='Plasma',
    colorbar_title="Temperature (°C)",
))

# Add dropdown menu
fig.update_layout(
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label=param.replace("_", " ").title(),
                    method='update',
                    args=[
                        {'z': [weather_data[param]]},
                        {'colorbar.title.text': param.replace("_", " ").title()}
                    ]
                )
                for param in weather_params
            ],
            direction='down',
            showactive=True,
            x=0.05,
            y=1.15,
            xanchor='left',
            yanchor='top'
        )
    ],
    title=dict(
        text=" Global Weather Conditions (Select Parameter)",
        x=0.5,
        font=dict(size=22, color='#0A043C')
    ),
    geo=dict(showframe=False, showcoastlines=True, projection_type='natural earth'),
    template='plotly_white',
)
fig.show()


### Global Weather Conditions (Interactive Map)

This interactive map visualizes the **average weather parameters** for each country.  
- Use the **dropdown menu** to switch between variables such as **Temperature**, **Humidity**, **Pressure**, and **Precipitation**.  
- **Darker shades** represent higher values for the selected parameter.  
- The visualization highlights **geographical patterns** in weather, showing which regions are hotter, wetter, or more humid.  

Overall, this map provides an engaging way to compare **climatic variations across countries**.


### Extreme Weather Events – Top 5 Hottest Days by Country

In [ ]:
import plotly.express as px

# -----------------------------------------------
# Top 5 Hottest Days by Country (Interactive)
# -----------------------------------------------

# Ensure 'date' is datetime
dataSelected['date'] = pd.to_datetime(dataSelected['date'])

# Find top 5 hottest days by country
top5_hottest = (
    dataSelected.loc[dataSelected.groupby('country')['temperature_celsius'].idxmax()]
    .sort_values(by='temperature_celsius', ascending=False)
    .head(5)
)

# Interactive bar chart
fig = px.bar(
    top5_hottest,
    x='country',
    y='temperature_celsius',
    color='temperature_celsius',
    text='location_name',
    hover_data=['date', 'location_name'],
    title='Top 5 Hottest Recorded Days by Country',
    color_continuous_scale='Reds'
)

fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Country',
    yaxis_title='Temperature (°C)',
    coloraxis_colorbar=dict(title='Temperature (°C)'),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(255,255,255,1)'
)

fig.show()


#### Top 5 Coldest Recorded Days

In [ ]:
import plotly.express as px

# -----------------------------------------------
# Top 5 Coldest Days by Country (Interactive)
# -----------------------------------------------

# Ensure 'date' is datetime
dataSelected['date'] = pd.to_datetime(dataSelected['date'])

# Find top 5 coldest days by country
top5_coldest = (
    dataSelected.loc[dataSelected.groupby('country')['temperature_celsius'].idxmin()]
    .sort_values(by='temperature_celsius', ascending=True)
    .head(5)
)

# Interactive bar chart
fig = px.bar(
    top5_coldest,
    x='country',
    y='temperature_celsius',
    color='temperature_celsius',
    text='location_name',
    hover_data=['date', 'location_name'],
    title='Top 5 Coldest Recorded Days by Country',
    color_continuous_scale='Blues_r'  # reversed blue scale for cold
)

fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Country',
    yaxis_title='Temperature (°C)',
    coloraxis_colorbar=dict(title='Temperature (°C)'),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(255,255,255,1)'
)

fig.show()


### Comparative Chart – Average Rainfall per Country

In [ ]:
import plotly.express as px

fig = px.bar(
    avg_rainfall.head(10),
    x='country',
    y='precip_mm',
    title='Top 10 Countries by Average Rainfall',
    color='precip_mm',
    color_continuous_scale='Blues'
)
fig.show()


### Summary and Insights

From all the visualizations, we can understand the overall **weather and air quality patterns** across different countries and months:

- Most regions experience **moderate temperatures**, with some showing seasonal peaks and drops.  
- **Humidity, wind speed, and pressure** remain fairly stable, while **precipitation** varies widely by country and season.  
- The **violin and distribution plots** show the spread and outliers in each weather variable, indicating occasional extreme conditions.  
- The **correlation heatmap** highlights relationships among weather features, showing how some factors move together.  
- **Air quality visualizations** reveal that pollutant levels fluctuate monthly, with certain pollutants peaking in specific seasons.  
- The **global interactive maps** help identify countries with higher pollution or distinct climate characteristics.

Overall, this analysis gives a clear view of **global climate behavior** and **air quality variations**, showing how environmental conditions change across time and geography.


In [ ]:
import pandas as pd
import plotly.express as px

# Calculate thresholds
temp_high = data['temperature_celsius'].mean() + 2 * data['temperature_celsius'].std()
rain_high = data['precip_mm'].mean() + 2 * data['precip_mm'].std()
wind_high = data['wind_kph'].mean() + 2 * data['wind_kph'].std()

# Filter extremes
extreme_temps = data[data['temperature_celsius'] > temp_high]
extreme_rain = data[data['precip_mm'] > rain_high]
extreme_wind = data[data['wind_kph'] > wind_high]

# Combine for visualization
extreme_events = pd.concat([
    extreme_temps.assign(event='High Temperature'),
    extreme_rain.assign(event='Heavy Rainfall'),
    extreme_wind.assign(event='High Wind')
])

# Scatter plot to visualize extreme events geographically
fig = px.scatter_geo(
    extreme_events,
    lat='latitude',
    lon='longitude',
    color='event',
    title=" Extreme Weather Events Around the World",
    hover_name='location_name',
    color_discrete_map={
        'High Temperature': '#FF4C4C',
        'Heavy Rainfall': '#3498DB',
        'High Wind': '#F1C40F'
    },
    template='plotly_white'
)
fig.update_traces(marker=dict(size=6, opacity=0.8))
fig.show()


### Extreme Weather Events

We identified events where temperature, rainfall, or wind exceeded two standard deviations above the mean — marking them as *extreme events*.

**Findings:**
- High-temperature events (heatwaves) occur mostly in tropical regions near the equator.  
- Heavy rainfall events are concentrated in coastal and monsoon zones.  
- High wind speeds are observed near coastal or desert regions, indicating storm-prone areas.

These anomalies highlight areas of potential climate risk and form the foundation for later dashboard insights.

